# Advanced Problems with Solutions: Metaclasses vs Class Decorators

This notebook is an advanced practice set built around a central Python metaprogramming distinction:

- **Class decorators** transform the specific class object they are applied to.
- **Metaclasses** participate in class creation and therefore naturally affect subclasses created with that metaclass.
- Custom metaclasses can create **metaclass conflicts** under inheritance unless their relationships are compatible.

The notebook goes beyond the introductory comparison with production-oriented patterns:
descriptor-safe wrapping, idempotence, opt-outs, `__init_subclass__`, decorator ordering,
metaclass lifecycle hooks, multiple inheritance, `ABCMeta` compatibility, class validation,
registries, instance creation control, and a capstone audit framework.

## How to use this notebook

For each problem:

1. Read the **Problem** section first.
2. Predict behavior before running the solution.
3. Run the solution cell.
4. Read the assertions as executable specifications.
5. Modify the code and test edge cases.

All intentional failure cases are caught so the notebook can run from top to bottom.

## Best-practice checklist

Throughout the solutions, prefer these habits:

- Use `functools.wraps` when wrapping functions.
- Avoid mutating inherited methods accidentally; inspect `vars(cls)` when you mean "defined on this class".
- Preserve descriptor types such as `staticmethod`, `classmethod`, and `property`.
- Mark wrappers so repeated instrumentation is **idempotent**.
- Provide explicit opt-outs for framework code.
- Keep metaclasses narrowly focused; prefer simpler tools when they are enough.
- Consider `__init_subclass__` before introducing a custom metaclass.
- Compose metaclasses carefully when using multiple inheritance or standard-library metaclasses such as `ABCMeta`.
- Test inheritance, overrides, descriptors, and duplicate application.

In [1]:
from __future__ import annotations

from abc import ABC, ABCMeta, abstractmethod
from functools import wraps
from typing import Callable
import inspect
import itertools

print("Python:", __import__("sys").version.split()[0])

Python: 3.13.7


# Problem 1 — Reproduce the inheritance blind spot of a class decorator

Create a method logger implemented as a **class decorator**.

Requirements:

- Log normal instance methods defined directly on the decorated class.
- Decorate `Base`.
- Create `Child(Base)` without decorating `Child`.
- Show that inherited wrapped methods still log, but methods newly defined on `Child` do not.

Then explain why.

In [2]:
def log_call(fn: Callable) -> Callable:
    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"[LOG] {fn.__qualname__} -> {result!r}")
        return result
    return wrapper


def basic_class_logger(cls):
    for name, obj in list(vars(cls).items()):
        if inspect.isfunction(obj):
            setattr(cls, name, log_call(obj))
    return cls


@basic_class_logger
class P1Base:
    def inherited_method(self):
        return "base method"


class P1Child(P1Base):
    def child_method(self):
        return "child method"


p1 = P1Child()

print("Calling inherited method:")
assert p1.inherited_method() == "base method"

print("\nCalling newly defined child method:")
assert p1.child_method() == "child method"

print("\nWrapped markers:")
print("P1Base.inherited_method has __wrapped__:", hasattr(P1Base.inherited_method, "__wrapped__"))
print("P1Child.child_method has __wrapped__:", hasattr(P1Child.child_method, "__wrapped__"))

assert hasattr(P1Base.inherited_method, "__wrapped__")
assert not hasattr(P1Child.child_method, "__wrapped__")

print("\nConclusion: the decorator transformed P1Base once; it did not run again when P1Child was created.")

Calling inherited method:
[LOG] P1Base.inherited_method -> 'base method'

Calling newly defined child method:

Wrapped markers:
P1Base.inherited_method has __wrapped__: True
P1Child.child_method has __wrapped__: False

Conclusion: the decorator transformed P1Base once; it did not run again when P1Child was created.


# Problem 2 — Make instrumentation propagate with a metaclass

Reimplement the previous behavior with a metaclass.

Requirements:

- Wrap functions defined in every class created by the metaclass.
- Define the metaclass only on the base class.
- Do **not** mention the metaclass in the subclass declaration.
- Prove that both base and child methods are wrapped.
- Inspect `type(Base)` and `type(Child)`.

In [3]:
class LoggingMeta(type):
    def __new__(mcls, name, bases, namespace):
        new_namespace = dict(namespace)

        for attr_name, obj in namespace.items():
            if inspect.isfunction(obj):
                new_namespace[attr_name] = log_call(obj)

        cls = super().__new__(mcls, name, bases, new_namespace)
        return cls


class P2Base(metaclass=LoggingMeta):
    def inherited_method(self):
        return "base via metaclass"


class P2Child(P2Base):
    def child_method(self):
        return "child via metaclass"


p2 = P2Child()

assert p2.inherited_method() == "base via metaclass"
assert p2.child_method() == "child via metaclass"

assert hasattr(P2Base.inherited_method, "__wrapped__")
assert hasattr(P2Child.child_method, "__wrapped__")

print("type(P2Base):", type(P2Base).__name__)
print("type(P2Child):", type(P2Child).__name__)
assert type(P2Base) is LoggingMeta
assert type(P2Child) is LoggingMeta

[LOG] P2Base.inherited_method -> 'base via metaclass'
[LOG] P2Child.child_method -> 'child via metaclass'
type(P2Base): LoggingMeta
type(P2Child): LoggingMeta


# Problem 3 — Build a descriptor-safe class transformer

A naïve `callable(obj)` or `inspect.isfunction(obj)` check misses important class attributes.

Build a reusable `instrument_attribute` helper that correctly handles:

- instance methods,
- `staticmethod`,
- `classmethod`,
- `property` getters/setters/deleters.

Preserve the descriptor type rather than replacing it with a plain function.

In [4]:
def logging_wrapper(fn: Callable) -> Callable:
    @wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"[TRACE] {fn.__qualname__} args={args[1:] if args else args} kwargs={kwargs} -> {result!r}")
        return result
    return wrapper


def instrument_attribute(obj):
    # Return a descriptor/function with its underlying callable(s) wrapped.
    if inspect.isfunction(obj):
        return logging_wrapper(obj)

    if isinstance(obj, staticmethod):
        return staticmethod(logging_wrapper(obj.__func__))

    if isinstance(obj, classmethod):
        return classmethod(logging_wrapper(obj.__func__))

    if isinstance(obj, property):
        return property(
            logging_wrapper(obj.fget) if obj.fget else None,
            logging_wrapper(obj.fset) if obj.fset else None,
            logging_wrapper(obj.fdel) if obj.fdel else None,
            doc=obj.__doc__,
        )

    return obj


def descriptor_safe_decorator(cls):
    for name, obj in list(vars(cls).items()):
        transformed = instrument_attribute(obj)
        if transformed is not obj:
            setattr(cls, name, transformed)
    return cls


@descriptor_safe_decorator
class P3Example:
    factor = 10

    def __init__(self, value):
        self._value = value

    def instance_method(self, n):
        return self._value + n

    @staticmethod
    def add(a, b):
        return a + b

    @classmethod
    def scale(cls, n):
        return cls.factor * n

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        self._value = new_value


x = P3Example(5)
assert x.instance_method(2) == 7
assert P3Example.add(2, 3) == 5
assert P3Example.scale(4) == 40
assert x.value == 5
x.value = 99
assert x.value == 99

raw = vars(P3Example)
assert isinstance(raw["add"], staticmethod)
assert isinstance(raw["scale"], classmethod)
assert isinstance(raw["value"], property)

print("Descriptor types preserved successfully.")

[TRACE] P3Example.__init__ args=(5,) kwargs={} -> None
[TRACE] P3Example.instance_method args=(2,) kwargs={} -> 7
[TRACE] P3Example.add args=(3,) kwargs={} -> 5
[TRACE] P3Example.scale args=(4,) kwargs={} -> 40
[TRACE] P3Example.value args=() kwargs={} -> 5
[TRACE] P3Example.value args=(99,) kwargs={} -> None
[TRACE] P3Example.value args=() kwargs={} -> 99
Descriptor types preserved successfully.


# Problem 4 — Make instrumentation idempotent

Frameworks often revisit classes. Applying a decorator twice must not create nested wrappers.

Implement a marker-based wrapper that:

- marks wrapped functions with `__instrumented__ = True`,
- returns the original wrapper unchanged if applied again,
- works through `staticmethod`, `classmethod`, and `property`,
- proves that a twice-decorated class logs each call only once.

In [5]:
def idempotent_wrapper(fn: Callable) -> Callable:
    if getattr(fn, "__instrumented__", False):
        return fn

    @wraps(fn)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return fn(*args, **kwargs)

    wrapper.calls = 0
    wrapper.__instrumented__ = True
    return wrapper


def instrument_attribute_idempotent(obj):
    if inspect.isfunction(obj):
        return idempotent_wrapper(obj)

    if isinstance(obj, staticmethod):
        return staticmethod(idempotent_wrapper(obj.__func__))

    if isinstance(obj, classmethod):
        return classmethod(idempotent_wrapper(obj.__func__))

    if isinstance(obj, property):
        return property(
            idempotent_wrapper(obj.fget) if obj.fget else None,
            idempotent_wrapper(obj.fset) if obj.fset else None,
            idempotent_wrapper(obj.fdel) if obj.fdel else None,
            doc=obj.__doc__,
        )

    return obj


def idempotent_class_decorator(cls):
    for name, obj in list(vars(cls).items()):
        transformed = instrument_attribute_idempotent(obj)
        if transformed is not obj:
            setattr(cls, name, transformed)
    return cls


@idempotent_class_decorator
@idempotent_class_decorator
class P4Service:
    def ping(self):
        return "pong"


service = P4Service()
assert service.ping() == "pong"

wrapped_ping = P4Service.ping
assert wrapped_ping.calls == 1
assert wrapped_ping.__instrumented__ is True
assert wrapped_ping.__name__ == "ping"

print("Call count:", wrapped_ping.calls)
print("Only one wrapper layer:", not getattr(wrapped_ping.__wrapped__, "__instrumented__", False))

Call count: 1
Only one wrapper layer: True


# Problem 5 — Add method-level opt-out controls

Instrumentation should be explicit and controllable.

Create:

- `@no_instrument` for methods that should never be wrapped,
- a metaclass that wraps other methods,
- support for instance, static, and class methods.

Test that the opt-out survives class creation.

In [6]:
def no_instrument(fn: Callable) -> Callable:
    fn.__no_instrument__ = True
    return fn


def should_skip_callable(fn: Callable) -> bool:
    return bool(getattr(fn, "__no_instrument__", False))


def optout_transform(obj):
    if inspect.isfunction(obj):
        return obj if should_skip_callable(obj) else idempotent_wrapper(obj)

    if isinstance(obj, staticmethod):
        fn = obj.__func__
        return obj if should_skip_callable(fn) else staticmethod(idempotent_wrapper(fn))

    if isinstance(obj, classmethod):
        fn = obj.__func__
        return obj if should_skip_callable(fn) else classmethod(idempotent_wrapper(fn))

    return obj


class OptOutLoggingMeta(type):
    def __new__(mcls, name, bases, namespace):
        new_ns = {
            key: optout_transform(value)
            for key, value in namespace.items()
        }
        return super().__new__(mcls, name, bases, new_ns)


class P5Worker(metaclass=OptOutLoggingMeta):
    def tracked(self):
        return "tracked"

    @no_instrument
    def raw(self):
        return "raw"

    @staticmethod
    @no_instrument
    def raw_static():
        return "raw static"

    @classmethod
    def tracked_class(cls):
        return cls.__name__


w = P5Worker()

assert getattr(P5Worker.tracked, "__instrumented__", False)
assert not getattr(P5Worker.raw, "__instrumented__", False)
assert not getattr(vars(P5Worker)["raw_static"].__func__, "__instrumented__", False)
assert getattr(vars(P5Worker)["tracked_class"].__func__, "__instrumented__", False)

print(w.tracked(), w.raw(), w.raw_static(), w.tracked_class())

tracked raw raw static P5Worker


# Problem 6 — Understand the metaclass lifecycle: `__new__`, `__init__`, and `__call__`

Instrument the metaclass itself to record events.

Predict the order of:

1. metaclass `__new__`,
2. metaclass `__init__`,
3. metaclass `__call__`,
4. instance `__new__`,
5. instance `__init__`.

Then verify the order with assertions.

In [7]:
P6_EVENTS = []


class LifecycleMeta(type):
    def __new__(mcls, name, bases, namespace):
        P6_EVENTS.append(("meta.__new__", name))
        return super().__new__(mcls, name, bases, namespace)

    def __init__(cls, name, bases, namespace):
        P6_EVENTS.append(("meta.__init__", name))
        super().__init__(name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        P6_EVENTS.append(("meta.__call__", cls.__name__))
        return super().__call__(*args, **kwargs)


class P6Thing(metaclass=LifecycleMeta):
    def __new__(cls, value):
        P6_EVENTS.append(("instance.__new__", cls.__name__))
        return super().__new__(cls)

    def __init__(self, value):
        P6_EVENTS.append(("instance.__init__", type(self).__name__))
        self.value = value


thing = P6Thing(123)

expected = [
    ("meta.__new__", "P6Thing"),
    ("meta.__init__", "P6Thing"),
    ("meta.__call__", "P6Thing"),
    ("instance.__new__", "P6Thing"),
    ("instance.__init__", "P6Thing"),
]

print(*P6_EVENTS, sep="\n")
assert P6_EVENTS == expected
assert thing.value == 123

('meta.__new__', 'P6Thing')
('meta.__init__', 'P6Thing')
('meta.__call__', 'P6Thing')
('instance.__new__', 'P6Thing')
('instance.__init__', 'P6Thing')


# Problem 7 — Decorator stacking order

Class decorators can be stacked freely, but order matters.

Create two decorators:

- one that adds a tag to the class,
- one that records which tag it sees.

Demonstrate that decorators are applied **bottom-up**.

In [8]:
P7_EVENTS = []


def add_tag(tag):
    def decorator(cls):
        cls.tag = tag
        P7_EVENTS.append(("add_tag", tag, cls.__name__))
        return cls
    return decorator


def observe_tag(cls):
    P7_EVENTS.append(("observe_tag", getattr(cls, "tag", None), cls.__name__))
    return cls


@observe_tag
@add_tag("inner-first")
class P7A:
    pass


@add_tag("outer-last")
@observe_tag
class P7B:
    pass


print(*P7_EVENTS, sep="\n")

assert P7_EVENTS[0] == ("add_tag", "inner-first", "P7A")
assert P7_EVENTS[1] == ("observe_tag", "inner-first", "P7A")
assert P7_EVENTS[2] == ("observe_tag", None, "P7B")
assert P7_EVENTS[3] == ("add_tag", "outer-last", "P7B")

('add_tag', 'inner-first', 'P7A')
('observe_tag', 'inner-first', 'P7A')
('observe_tag', None, 'P7B')
('add_tag', 'outer-last', 'P7B')


# Problem 8 — Use `__init_subclass__` as a lighter alternative

A metaclass is not always necessary.

Build a base class that automatically instruments every subclass via `__init_subclass__`.

Requirements:

- no custom metaclass,
- child and grandchild methods are instrumented,
- inherited methods are not wrapped repeatedly,
- `super().__init_subclass__()` is called.

In [9]:
class AutoInstrument:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        for name, obj in list(vars(cls).items()):
            transformed = instrument_attribute_idempotent(obj)
            if transformed is not obj:
                setattr(cls, name, transformed)


class P8Base(AutoInstrument):
    def base_method(self):
        return "base"


class P8Child(P8Base):
    def child_method(self):
        return "child"


class P8GrandChild(P8Child):
    def grandchild_method(self):
        return "grandchild"


g = P8GrandChild()

assert g.base_method() == "base"
assert g.child_method() == "child"
assert g.grandchild_method() == "grandchild"

assert P8Base.base_method.calls == 1
assert P8Child.child_method.calls == 1
assert P8GrandChild.grandchild_method.calls == 1

assert P8Child.base_method is P8Base.base_method
assert P8GrandChild.child_method is P8Child.child_method

print("__init_subclass__ propagated behavior without a custom metaclass.")

__init_subclass__ propagated behavior without a custom metaclass.


# Problem 9 — Diagnose and resolve a metaclass conflict

Create two unrelated custom metaclasses and two base classes that use them.

First, confirm that inheriting from both base classes raises `TypeError`.

Then resolve the conflict with a **composite metaclass** that subclasses both metaclasses.

In [10]:
class MetaA(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        namespace.setdefault("from_meta_a", True)
        return super().__new__(mcls, name, bases, namespace)


class MetaB(type):
    def __new__(mcls, name, bases, namespace):
        namespace = dict(namespace)
        namespace.setdefault("from_meta_b", True)
        return super().__new__(mcls, name, bases, namespace)


class P9A(metaclass=MetaA):
    pass


class P9B(metaclass=MetaB):
    pass


try:
    class P9Broken(P9A, P9B):
        pass
except TypeError as exc:
    conflict_message = str(exc)
    print("Expected conflict:", conflict_message)
else:
    raise AssertionError("Expected a metaclass conflict")


class MetaAB(MetaA, MetaB):
    pass


class P9Resolved(P9A, P9B, metaclass=MetaAB):
    pass


assert P9Resolved.from_meta_a is True
assert P9Resolved.from_meta_b is True
assert issubclass(type(P9Resolved), MetaA)
assert issubclass(type(P9Resolved), MetaB)

print("Resolved metaclass:", type(P9Resolved).__name__)
print("Metaclass MRO:", [c.__name__ for c in type(P9Resolved).__mro__])

Expected conflict: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases
Resolved metaclass: MetaAB
Metaclass MRO: ['MetaAB', 'MetaA', 'MetaB', 'type', 'object']


# Problem 10 — Compose safely with `ABCMeta`

Abstract base classes already use `ABCMeta`.

Create an instrumentation metaclass that is compatible with an abstract base class by subclassing `ABCMeta`.

Requirements:

- abstract methods remain abstract,
- concrete subclass methods are instrumented,
- instantiating the abstract base fails,
- instantiating the concrete subclass succeeds.

In [11]:
class InstrumentedABCMeta(ABCMeta):
    def __new__(mcls, name, bases, namespace):
        new_ns = {}
        for key, value in namespace.items():
            if inspect.isfunction(value):
                new_ns[key] = idempotent_wrapper(value)
            else:
                new_ns[key] = value
        return super().__new__(mcls, name, bases, new_ns)


class P10Processor(ABC, metaclass=InstrumentedABCMeta):
    @abstractmethod
    def process(self, value):
        raise NotImplementedError


assert inspect.isabstract(P10Processor)

try:
    P10Processor()
except TypeError as exc:
    print("Expected abstract-class error:", exc)
else:
    raise AssertionError("Abstract base class should not be instantiable")


class P10Concrete(P10Processor):
    def process(self, value):
        return value * 2


processor = P10Concrete()
assert processor.process(21) == 42
assert getattr(P10Concrete.process, "__instrumented__", False)
assert not inspect.isabstract(P10Concrete)

print("ABCMeta compatibility preserved.")

Expected abstract-class error: Can't instantiate abstract class P10Processor without an implementation for abstract method 'process'
ABCMeta compatibility preserved.


# Problem 11 — Enforce a class-level contract at class creation time

Use a metaclass to validate classes.

Contract:

- Every non-abstract plugin class must define a non-empty string `plugin_name`.
- Every plugin class must define a callable `run`.
- An abstract framework base may omit `plugin_name`.

Catch invalid class definitions and show helpful error messages.

In [12]:
class PluginContractMeta(ABCMeta):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if inspect.isabstract(cls):
            return cls

        plugin_name = getattr(cls, "plugin_name", None)
        if not isinstance(plugin_name, str) or not plugin_name.strip():
            raise TypeError(
                f"{name}: concrete plugins must define a non-empty string 'plugin_name'"
            )

        run = getattr(cls, "run", None)
        if not callable(run):
            raise TypeError(f"{name}: concrete plugins must define callable 'run'")

        return cls


class P11Plugin(ABC, metaclass=PluginContractMeta):
    @abstractmethod
    def run(self, payload):
        raise NotImplementedError


class P11Upper(P11Plugin):
    plugin_name = "upper"

    def run(self, payload):
        return str(payload).upper()


assert P11Upper().run("hello") == "HELLO"


def make_invalid_plugin_missing_name():
    class MissingName(P11Plugin):
        def run(self, payload):
            return payload
    return MissingName


try:
    make_invalid_plugin_missing_name()
except TypeError as exc:
    print("Expected validation error:", exc)
else:
    raise AssertionError("MissingName should fail class creation")

Expected validation error: MissingName: concrete plugins must define a non-empty string 'plugin_name'


# Problem 12 — Build a concrete-subclass registry

Create a metaclass that automatically registers **concrete** subclasses by a unique key.

Requirements:

- abstract classes are not registered,
- concrete classes register automatically,
- duplicate keys fail immediately,
- registry lookup returns the class,
- subclasses continue using the registry metaclass automatically.

In [13]:
class RegistryMeta(ABCMeta):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if not inspect.isabstract(cls):
            key = getattr(cls, "kind", None)
            if not isinstance(key, str) or not key:
                raise TypeError(f"{name}: concrete registered classes need a non-empty 'kind'")

            if key in mcls.registry:
                other = mcls.registry[key]
                raise TypeError(
                    f"{name}: duplicate kind {key!r}; already registered by {other.__name__}"
                )

            mcls.registry[key] = cls

        return cls


class P12Command(ABC, metaclass=RegistryMeta):
    @abstractmethod
    def execute(self):
        raise NotImplementedError


class P12Start(P12Command):
    kind = "start"

    def execute(self):
        return "started"


class P12Stop(P12Command):
    kind = "stop"

    def execute(self):
        return "stopped"


assert RegistryMeta.registry["start"] is P12Start
assert RegistryMeta.registry["stop"] is P12Stop
assert RegistryMeta.registry["start"]().execute() == "started"

try:
    class P12Duplicate(P12Command):
        kind = "start"

        def execute(self):
            return "duplicate"
except TypeError as exc:
    print("Expected duplicate-key error:", exc)
else:
    raise AssertionError("Duplicate registry key should fail")

print("Registry:", {k: v.__name__ for k, v in RegistryMeta.registry.items()})

Expected duplicate-key error: P12Duplicate: duplicate kind 'start'; already registered by P12Start
Registry: {'start': 'P12Start', 'stop': 'P12Stop'}


# Problem 13 — Control instance creation with metaclass `__call__`

Use `__call__` on the metaclass to implement a **per-class instance counter**.

Requirements:

- every successful construction increments a counter,
- subclasses get independent counters,
- failed construction must not increment,
- expose the count as `Class.instances_created`.

In [14]:
class CountingMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.instances_created = 0
        return cls

    def __call__(cls, *args, **kwargs):
        obj = super().__call__(*args, **kwargs)
        cls.instances_created += 1
        return obj


class P13Connection(metaclass=CountingMeta):
    def __init__(self, endpoint):
        if not endpoint:
            raise ValueError("endpoint is required")
        self.endpoint = endpoint


class P13SecureConnection(P13Connection):
    pass


a = P13Connection("db://one")
b = P13Connection("db://two")
c = P13SecureConnection("tls://one")

assert P13Connection.instances_created == 2
assert P13SecureConnection.instances_created == 1

try:
    P13Connection("")
except ValueError:
    pass

assert P13Connection.instances_created == 2

print("P13Connection:", P13Connection.instances_created)
print("P13SecureConnection:", P13SecureConnection.instances_created)

P13Connection: 2
P13SecureConnection: 1


# Problem 14 — Compare three propagation strategies

Implement the same "wrap methods on subclasses" goal three ways:

1. explicit class decorators,
2. a metaclass,
3. `__init_subclass__`.

Create two generations of subclasses and compare the amount of repeated declaration required.

The goal is not to prove one tool is universally best; it is to see their trade-offs in executable form.

In [15]:
@idempotent_class_decorator
class P14DecoratedBase:
    def base(self):
        return "base"


@idempotent_class_decorator
class P14DecoratedChild(P14DecoratedBase):
    def child(self):
        return "child"


@idempotent_class_decorator
class P14DecoratedGrandChild(P14DecoratedChild):
    def grand(self):
        return "grand"


class P14Meta(type):
    def __new__(mcls, name, bases, namespace):
        ns = {
            key: instrument_attribute_idempotent(value)
            for key, value in namespace.items()
        }
        return super().__new__(mcls, name, bases, ns)


class P14MetaBase(metaclass=P14Meta):
    def base(self):
        return "base"


class P14MetaChild(P14MetaBase):
    def child(self):
        return "child"


class P14MetaGrandChild(P14MetaChild):
    def grand(self):
        return "grand"


class P14HookBase:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        for key, value in list(vars(cls).items()):
            new_value = instrument_attribute_idempotent(value)
            if new_value is not value:
                setattr(cls, key, new_value)


class P14HookChild(P14HookBase):
    def child(self):
        return "child"


class P14HookGrandChild(P14HookChild):
    def grand(self):
        return "grand"


assert getattr(P14DecoratedGrandChild.grand, "__instrumented__", False)
assert getattr(P14MetaGrandChild.grand, "__instrumented__", False)
assert getattr(P14HookGrandChild.grand, "__instrumented__", False)

print("All three can work.")
print("Decorator strategy repeats syntax on each class.")
print("Metaclass and __init_subclass__ propagate through inheritance.")

All three can work.
Decorator strategy repeats syntax on each class.
Metaclass and __init_subclass__ propagate through inheritance.


# Problem 15 — Advanced metaclass cooperation with `super()`

A composite metaclass only works well when participating metaclasses are **cooperative**.

Build two metaclasses:

- one adds a sequence number to each class,
- one records every class name created.

Both must call `super().__new__`.

Then compose them and prove that both behaviors run for a subclass.

In [16]:
P15_CREATED_NAMES = []
P15_SEQUENCE = itertools.count(1)


class SequencedMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.creation_sequence = next(P15_SEQUENCE)
        return cls


class RecordingMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        P15_CREATED_NAMES.append(name)
        return cls


class CooperativeMeta(SequencedMeta, RecordingMeta):
    pass


class P15Base(metaclass=CooperativeMeta):
    pass


class P15Child(P15Base):
    pass


assert P15_CREATED_NAMES[-2:] == ["P15Base", "P15Child"]
assert P15Base.creation_sequence < P15Child.creation_sequence

print("Recorded:", P15_CREATED_NAMES[-2:])
print("Sequences:", P15Base.creation_sequence, P15Child.creation_sequence)
print("Composite MRO:", [c.__name__ for c in CooperativeMeta.__mro__])

Recorded: ['P15Base', 'P15Child']
Sequences: 1 2
Composite MRO: ['CooperativeMeta', 'SequencedMeta', 'RecordingMeta', 'type', 'object']


# Problem 16 — Prevent accidental wrapping of special methods

Blindly wrapping every function can make logs noisy and interfere with framework internals.

Create a policy that:

- wraps public methods by default,
- skips dunder methods,
- allows specific dunder names through an allow-list,
- skips private helper methods starting with `_`,
- demonstrates that policy decisions are centralized.

In [17]:
def should_instrument_name(name: str, *, allow_dunder=frozenset()):
    if name in allow_dunder:
        return True
    if name.startswith("__") and name.endswith("__"):
        return False
    if name.startswith("_"):
        return False
    return True


def selective_class_decorator(cls, *, allow_dunder=frozenset()):
    for name, obj in list(vars(cls).items()):
        if should_instrument_name(name, allow_dunder=allow_dunder):
            new_obj = instrument_attribute_idempotent(obj)
            if new_obj is not obj:
                setattr(cls, name, new_obj)
    return cls


def instrument_publics(*, allow_dunder=frozenset()):
    return lambda cls: selective_class_decorator(cls, allow_dunder=allow_dunder)


@instrument_publics(allow_dunder={"__len__"})
class P16Collection:
    def __init__(self, values):
        self._values = list(values)

    def public(self):
        return len(self._values)

    def _helper(self):
        return "private"

    def __len__(self):
        return len(self._values)


obj16 = P16Collection([1, 2, 3])

assert getattr(P16Collection.public, "__instrumented__", False)
assert not getattr(P16Collection._helper, "__instrumented__", False)
assert getattr(P16Collection.__len__, "__instrumented__", False)
assert not getattr(P16Collection.__init__, "__instrumented__", False)
assert len(obj16) == 3

print("Central policy works.")

Central policy works.


# Problem 17 — Capstone: production-style audit metaclass

Build an `AuditMeta` framework with all of the following:

- automatic inheritance propagation,
- instance / static / class method support,
- property getter/setter support,
- `@audit_ignore` opt-out,
- idempotent wrapping,
- configurable class-level prefix,
- public-method-only policy,
- structured in-memory audit events,
- tests for inheritance and overrides.

This combines the main techniques from the notebook.

In [18]:
AUDIT_EVENTS = []


def audit_ignore(fn: Callable) -> Callable:
    fn.__audit_ignore__ = True
    return fn


def make_audit_wrapper(fn: Callable, *, owner_name: str, attr_name: str, prefix: str):
    if getattr(fn, "__audit_wrapped__", False):
        return fn

    if getattr(fn, "__audit_ignore__", False):
        return fn

    @wraps(fn)
    def wrapper(*args, **kwargs):
        try:
            result = fn(*args, **kwargs)
        except Exception as exc:
            AUDIT_EVENTS.append({
                "prefix": prefix,
                "owner": owner_name,
                "method": attr_name,
                "status": "error",
                "error_type": type(exc).__name__,
            })
            raise
        else:
            AUDIT_EVENTS.append({
                "prefix": prefix,
                "owner": owner_name,
                "method": attr_name,
                "status": "ok",
                "result_type": type(result).__name__,
            })
            return result

    wrapper.__audit_wrapped__ = True
    return wrapper


def audit_transform(obj, *, owner_name: str, attr_name: str, prefix: str):
    if inspect.isfunction(obj):
        return make_audit_wrapper(
            obj, owner_name=owner_name, attr_name=attr_name, prefix=prefix
        )

    if isinstance(obj, staticmethod):
        return staticmethod(
            make_audit_wrapper(
                obj.__func__,
                owner_name=owner_name,
                attr_name=attr_name,
                prefix=prefix,
            )
        )

    if isinstance(obj, classmethod):
        return classmethod(
            make_audit_wrapper(
                obj.__func__,
                owner_name=owner_name,
                attr_name=attr_name,
                prefix=prefix,
            )
        )

    if isinstance(obj, property):
        return property(
            make_audit_wrapper(
                obj.fget,
                owner_name=owner_name,
                attr_name=f"{attr_name}.get",
                prefix=prefix,
            ) if obj.fget else None,
            make_audit_wrapper(
                obj.fset,
                owner_name=owner_name,
                attr_name=f"{attr_name}.set",
                prefix=prefix,
            ) if obj.fset else None,
            make_audit_wrapper(
                obj.fdel,
                owner_name=owner_name,
                attr_name=f"{attr_name}.del",
                prefix=prefix,
            ) if obj.fdel else None,
            doc=obj.__doc__,
        )

    return obj


class AuditMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        prefix = namespace.get("audit_prefix")

        if prefix is None:
            prefix = next(
                (getattr(base, "audit_prefix") for base in bases if hasattr(base, "audit_prefix")),
                "AUDIT",
            )

        new_ns = dict(namespace)
        new_ns["audit_prefix"] = prefix

        for attr_name, obj in namespace.items():
            if not should_instrument_name(attr_name):
                continue

            new_ns[attr_name] = audit_transform(
                obj,
                owner_name=name,
                attr_name=attr_name,
                prefix=prefix,
            )

        return super().__new__(mcls, name, bases, new_ns)


class P17Account(metaclass=AuditMeta):
    audit_prefix = "BANK"

    def __init__(self, balance):
        self._balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("amount must be positive")
        self._balance += amount
        return self._balance

    @property
    def balance(self):
        return self._balance

    @audit_ignore
    def healthcheck(self):
        return "ok"

    @staticmethod
    def currency():
        return "USD"

    @classmethod
    def category(cls):
        return cls.__name__


class P17Savings(P17Account):
    def add_interest(self, rate):
        self._balance *= (1 + rate)
        return round(self._balance, 2)

    def deposit(self, amount):
        return super().deposit(amount)


AUDIT_EVENTS.clear()

acct = P17Savings(100)
assert acct.deposit(50) == 150
assert acct.add_interest(0.10) == 165.0
assert acct.balance == 165.0
assert acct.healthcheck() == "ok"
assert acct.currency() == "USD"
assert acct.category() == "P17Savings"

try:
    acct.deposit(-1)
except ValueError:
    pass

assert not any(event["method"] == "healthcheck" for event in AUDIT_EVENTS)

assert any(
    event["owner"] == "P17Savings"
    and event["method"] == "deposit"
    and event["status"] == "ok"
    for event in AUDIT_EVENTS
)

assert any(event["method"] == "add_interest" for event in AUDIT_EVENTS)
assert any(event["method"] == "balance.get" for event in AUDIT_EVENTS)
assert any(event["status"] == "error" and event["method"] == "deposit" for event in AUDIT_EVENTS)

print("Audit events:")
for event in AUDIT_EVENTS:
    print(event)

Audit events:
{'prefix': 'BANK', 'owner': 'P17Account', 'method': 'deposit', 'status': 'ok', 'result_type': 'int'}
{'prefix': 'BANK', 'owner': 'P17Savings', 'method': 'deposit', 'status': 'ok', 'result_type': 'int'}
{'prefix': 'BANK', 'owner': 'P17Savings', 'method': 'add_interest', 'status': 'ok', 'result_type': 'float'}
{'prefix': 'BANK', 'owner': 'P17Account', 'method': 'balance.get', 'status': 'ok', 'result_type': 'float'}
{'prefix': 'BANK', 'owner': 'P17Account', 'method': 'currency', 'status': 'ok', 'result_type': 'str'}
{'prefix': 'BANK', 'owner': 'P17Account', 'method': 'category', 'status': 'ok', 'result_type': 'str'}
{'prefix': 'BANK', 'owner': 'P17Account', 'method': 'deposit', 'status': 'error', 'error_type': 'ValueError'}
{'prefix': 'BANK', 'owner': 'P17Savings', 'method': 'deposit', 'status': 'error', 'error_type': 'ValueError'}


# Synthesis — Choosing the right tool

Use a **class decorator** when:

- transformation is local and explicit,
- you want easy stacking/composition,
- you do not need automatic behavior on every subclass,
- the class hierarchy is not owned by your framework.

Use **`__init_subclass__`** when:

- behavior should propagate through a hierarchy you control,
- subclass initialization is enough,
- you want something lighter than a metaclass.

Use a **metaclass** when:

- behavior must participate in class creation itself,
- you need inheritance-wide enforcement or registration,
- you need control through metaclass hooks such as `__new__`, `__init__`, or `__call__`,
- class-level contracts must fail at definition time.

Watch for:

- metaclass conflicts,
- descriptor handling,
- wrapper duplication,
- abstract-method metadata,
- decorator order,
- hidden effects that make debugging harder.

The strongest design is usually the **simplest mechanism that satisfies the propagation and lifecycle requirements**.

# Extra challenge ideas

Try these without looking up a complete implementation first:

1. Extend `AuditMeta` so a subclass can set `audit_enabled = False`.
2. Add asynchronous function support and correctly `await` coroutine methods.
3. Add timing with `time.perf_counter()` while keeping wrappers idempotent.
4. Make the registry in Problem 12 support namespaced keys such as `"storage:s3"`.
5. Combine the registry and audit metaclasses cooperatively.
6. Add a class decorator that returns a **new subclass** instead of mutating the original class. Compare identity and `isinstance` behavior.
7. Add tests proving `functools.wraps` preserves `__name__`, `__doc__`, `__qualname__`, and `inspect.signature`.
8. Create a deliberately non-cooperative metaclass that forgets `super()` and observe which composite behavior disappears.